# Delivery Time Prediction — Data Cleaning

**Steps (exactly):**
1. Drop specified columns
2. Drop rows with missing values
3. Clip target at p99
4. Log-transform heavy-tailed numeric features

**Output:** `train_clean.csv` / `test_clean.csv`

---
## 0. Setup

In [41]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

TARGET = 'delivery_time_days'

DROP_COLS = [ 
    'id',
    'order_id', 'customer_unique_id', 'product_id',
    'product_name_length', 'product_description_length',
    'product_length_cm', 'product_height_cm', 'product_width_cm', 
    'product_photos_qty'
]

LOG_COLS = [
    'price', 'freight_value', 'product_weight_g',
    'volume_cm3', 'quantity',
]

print('Setup complete ✓')

Setup complete ✓


---
## 1. Load data

In [42]:
train = pd.read_csv('dataset/train.csv')
test  = pd.read_csv('dataset/test.csv')

print(f'Train : {train.shape}')
print(f'Test  : {test.shape}')

Train : (85159, 26)
Test  : (15029, 25)


---
## 2. Drop columns

In [43]:
train = train.drop(columns=[c for c in DROP_COLS if c in train.columns])
test  = test.drop(columns=[c for c in DROP_COLS if c in test.columns])

print(f'Train : {train.shape}')
print(f'Test  : {test.shape}')

Train : (85159, 16)
Test  : (15029, 15)


---
## 3. Drop rows with missing values

In [44]:
n_train_before = len(train)


train = train.dropna().reset_index(drop=True)  # drop is fine for train
print(f'Train : {n_train_before:,} → {len(train):,}  (dropped {n_train_before - len(train):,})')

# For test: impute with train medians instead
for col in test.columns:
    if test[col].isnull().any():
        if test[col].dtype == 'object':
            test[col].fillna(train[col].mode()[0], inplace=True)
        else:
            test[col].fillna(train[col].median(), inplace=True)

assert test.isnull().sum().sum() == 0, 'Still missing values in test!'
print(f'Test shape preserved: {test.shape}')

assert train.isnull().sum().sum() == 0, 'Missing values remain in train!'

print('✓ No missing values remain')

Train : 85,159 → 83,401  (dropped 1,758)
Test shape preserved: (15029, 15)
✓ No missing values remain


---
## 4. Clip target at p99

In [45]:
p99 = train[TARGET].quantile(0.99)
n_clipped = (train[TARGET] > p99).sum()

train[TARGET] = train[TARGET].clip(upper=p99)

print(f'p99 threshold : {p99:.2f} days')
print(f'Rows clipped  : {n_clipped} ({n_clipped / len(train) * 100:.2f}%)')
print(f'New max       : {train[TARGET].max():.2f} days')

p99 threshold : 47.22 days
Rows clipped  : 834 (1.00%)
New max       : 47.22 days


---
## 5. Log-transform heavy-tailed numeric features

In [46]:
log_cols_present = [c for c in LOG_COLS if c in train.columns]

for col in log_cols_present:
    for df in (train, test):
        df[col] = np.log1p(df[col].clip(lower=0))

print(f'log1p applied to: {log_cols_present}')

log1p applied to: ['price', 'freight_value', 'product_weight_g', 'volume_cm3', 'quantity']


---
## 6. Validation

In [47]:
assert train.isnull().sum().sum() == 0,           'Missing values in train!'
assert train[TARGET].max() <= p99,                'Target not clipped!'
assert np.isinf(train.select_dtypes(include=np.number).values).sum() == 0, 'Inf values in train!'

assert TARGET not in test.columns,                'Target leaking into test!'

print('All checks passed ✓')
print(f'\nFinal shapes — Train: {train.shape}  |  Test: {test.shape}')
print(f'Columns: {list(train.columns)}')

All checks passed ✓

Final shapes — Train: (83401, 16)  |  Test: (15029, 15)
Columns: ['customer_city', 'customer_lat', 'customer_lng', 'seller_id', 'seller_city', 'seller_lat', 'seller_lng', 'quantity', 'price', 'freight_value', 'product_weight_g', 'volume_cm3', 'product_category_name_english', 'order_purchase_timestamp', 'order_approved_at', 'delivery_time_days']


---
## 7. Save

In [48]:
import os
os.makedirs('dataset/processed', exist_ok=True)

train.to_csv('dataset/processed/train_clean.csv', index=False)
test.to_csv( 'dataset/processed/test_clean.csv',  index=False)

print('Saved train_clean.csv✓')
print('Saved test_clean.csv  ✓')

Saved train_clean.csv✓
Saved test_clean.csv  ✓
